# Notebook 2: Data Wrangling

Loads the raw CSVs from Notebook 1, cleans and joins them, engineers features, and saves the final dataset.

**Pipeline:**
1. Load & concat 5 seasons of game logs with Polars
2. Clean game logs (types, DNP filter)
3. Clean salaries (types, dedup traded players)
4. Entity linking — normalize player names + fuzzy match
5. SQL join via DuckDB
6. Feature engineering (per-36, usage, targets)
7. Save to `../data/processed/`

**Expected output:** `nba_player_games_clean.parquet` with ~115,000 rows × ~45 columns

In [39]:
import math
import re
from pathlib import Path

import polars as pl
import duckdb
from rapidfuzz import process as rf_process, fuzz

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

# Columns to keep from game logs (drop rank/ordering columns)
KEEP_COLS = [
    "SEASON", "PLAYER_ID", "PLAYER_NAME", "TEAM_ABBREVIATION",
    "GAME_ID", "GAME_DATE", "MATCHUP", "WL",
    "MIN", "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB",
    "AST", "TOV", "STL", "BLK", "BLKA", "PF", "PFD",
    "PTS", "PLUS_MINUS",
    "NBA_FANTASY_PTS",
]

print(f"Polars version: {pl.__version__}")
print(f"DuckDB version: {duckdb.__version__}")

Polars version: 1.39.3
DuckDB version: 1.5.1


## Step 1: Load & Concat Game Logs

In [40]:
frames = []
for season in SEASONS:
    path = RAW_DIR / f"game_logs_{season}.csv"
    df = pl.read_csv(path, infer_schema_length=5000)
    # Select only columns that exist in this file (schema may vary slightly)
    available = [c for c in KEEP_COLS if c in df.columns]
    frames.append(df.select(available))
    print(f"  Loaded {season}: {len(df):,} rows")

game_logs_raw = pl.concat(frames, how="diagonal_relaxed")
print(f"\nCombined: {game_logs_raw.shape[0]:,} rows × {game_logs_raw.shape[1]} cols")

  Loaded 2020-21: 23,054 rows
  Loaded 2021-22: 26,039 rows
  Loaded 2022-23: 25,894 rows
  Loaded 2023-24: 26,401 rows
  Loaded 2024-25: 26,306 rows

Combined: 127,694 rows × 31 cols


## Step 2: Clean Game Logs

- Parse `GAME_DATE` to Date type
- Cast numeric stats to Float32
- Filter out DNP (Did Not Play) rows where `MIN` is null or 0

**Why Float32?** Saves memory — ~50% vs Float64 for ~115k rows × 20+ numeric columns.

In [41]:
NUMERIC_STAT_COLS = [
    "MIN", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB",
    "AST", "TOV", "STL", "BLK", "BLKA", "PF", "PFD",
    "PTS", "PLUS_MINUS", "NBA_FANTASY_PTS",
]

# Only cast columns that actually exist
existing_numeric = [c for c in NUMERIC_STAT_COLS if c in game_logs_raw.columns]

game_logs = (
    game_logs_raw
    .with_columns(
        # nba_api returns dates as "2024-04-14T00:00:00" (ISO datetime with time component)
        pl.col("GAME_DATE").str.strptime(pl.Date, "%Y-%m-%dT%H:%M:%S", strict=False).alias("GAME_DATE"),
        *[pl.col(c).cast(pl.Float32, strict=False) for c in existing_numeric],
    )
    # Filter DNP rows: MIN must be non-null and > 0
    .filter(pl.col("MIN").is_not_null() & (pl.col("MIN") > 0))
)

print(f"After DNP filter: {len(game_logs):,} rows (removed {len(game_logs_raw) - len(game_logs):,} DNP rows)")
print(f"Date range: {game_logs['GAME_DATE'].min()} → {game_logs['GAME_DATE'].max()}")
print(f"Unique players: {game_logs['PLAYER_NAME'].n_unique():,}")
print(f"Unique seasons: {sorted(game_logs['SEASON'].unique().to_list())}")

After DNP filter: 127,684 rows (removed 10 DNP rows)
Date range: 2020-12-22 → 2025-04-13
Unique players: 991
Unique seasons: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


## Step 3: Load & Clean Salaries

In [42]:
salary_frames = []
for season in SEASONS:
    path = RAW_DIR / f"salaries_{season}.csv"
    df = pl.read_csv(path)
    salary_frames.append(df)
    print(f"  Loaded salaries_{season}.csv: {len(df):,} rows")

salaries_raw = pl.concat(salary_frames, how="diagonal_relaxed")
print(f"\nCombined salaries: {len(salaries_raw):,} rows")

# Clean: drop zero/null salaries (two-way contract placeholders)
salaries = (
    salaries_raw
    .with_columns(pl.col("SALARY").cast(pl.Float64, strict=False))
    .filter(pl.col("SALARY").is_not_null() & (pl.col("SALARY") > 0))
)

print(f"After cleaning: {len(salaries):,} rows")
print(f"Salary range: ${salaries['SALARY'].min():,.0f} — ${salaries['SALARY'].max():,.0f}")

  Loaded salaries_2020-21.csv: 574 rows
  Loaded salaries_2021-22.csv: 657 rows
  Loaded salaries_2022-23.csv: 511 rows
  Loaded salaries_2023-24.csv: 531 rows
  Loaded salaries_2024-25.csv: 518 rows

Combined salaries: 2,791 rows
After cleaning: 2,677 rows
Salary range: $5,318 — $55,761,216


## Step 3b: Load Player Bios

Join position, birth date, height, weight, draft info, and first NBA season onto the game logs. Age at game time and years of experience are computed as derived features in Step 6.

In [43]:
bios = pl.read_csv(RAW_DIR / "player_bios.csv")

# Parse birth date (BBRef format: "July 1, 1989")
bios = bios.with_columns(
    pl.col("BIRTH_DATE").str.strptime(pl.Date, "%B %d, %Y", strict=False).alias("BIRTH_DATE_DT"),
    # Simplify position: keep first listed (e.g. "SF-PF" → "SF")
    pl.col("POSITION").str.split("-").list.first().str.strip_chars().alias("POSITION_SIMPLE"),
    # Parse years experience: "R" means rookie (0 years)
    pl.col("YEARS_EXPERIENCE").replace({"R": "0"}).cast(pl.Int16, strict=False).alias("YEARS_EXP"),
)

print(f"Bios loaded: {len(bios):,} players")
print(f"Positions: {sorted(bios['POSITION_SIMPLE'].unique().to_list())}")
print(bios.select(['BBREF_ID','PLAYER_NAME','POSITION_SIMPLE','BIRTH_DATE_DT','YEARS_EXP','HEIGHT','WEIGHT']).head(5))

Bios loaded: 569 players
Positions: ['C', 'PF', 'PG', 'SF', 'SG']
shape: (5, 7)
┌───────────┬──────────────────┬─────────────────┬───────────────┬───────────┬────────┬────────┐
│ BBREF_ID  ┆ PLAYER_NAME      ┆ POSITION_SIMPLE ┆ BIRTH_DATE_DT ┆ YEARS_EXP ┆ HEIGHT ┆ WEIGHT │
│ ---       ┆ ---              ┆ ---             ┆ ---           ┆ ---       ┆ ---    ┆ ---    │
│ str       ┆ str              ┆ str             ┆ date          ┆ i16       ┆ str    ┆ i64    │
╞═══════════╪══════════════════╪═════════════════╪═══════════════╪═══════════╪════════╪════════╡
│ millele01 ┆ Leonard Miller   ┆ SF              ┆ 2003-11-26    ┆ 1         ┆ 6-10   ┆ 220    │
│ martity01 ┆ Tyrese Martin    ┆ SG              ┆ 1999-03-07    ┆ 1         ┆ 6-6    ┆ 215    │
│ bookede01 ┆ Devin Booker     ┆ SG              ┆ 1996-10-30    ┆ 9         ┆ 6-5    ┆ 206    │
│ thybuma01 ┆ Matisse Thybulle ┆ SG              ┆ 1997-03-04    ┆ 5         ┆ 6-5    ┆ 202    │
│ willije02 ┆ Nate Williams    ┆ SG            

## Step 4: Entity Linking — Name Normalization

nba_api and Basketball-Reference use slightly different player name conventions (e.g. "Marcus Morris Sr." vs "Marcus Morris", "Bones Hyland" vs legal name). We normalize names to a common form, then use fuzzy matching as a fallback.

**Strategy:**
1. Lowercase + strip whitespace
2. Remove suffixes (Jr., Sr., II, III, IV, V)
3. Remove punctuation
4. Apply manual override dict for known nickname mismatches
5. rapidfuzz fuzzy fallback (token_sort_ratio ≥ 88)

In [44]:
def normalize_name(name: str) -> str:
    """Normalize player name for cross-source matching."""
    name = name.lower().strip()
    # Remove suffixes
    name = re.sub(r"\s+(jr\.?|sr\.?|ii|iii|iv|v)$", "", name, flags=re.IGNORECASE)
    # Remove punctuation (apostrophes, hyphens, periods)
    name = re.sub(r"[^\w\s]", "", name)
    # Collapse whitespace
    return re.sub(r"\s+", " ", name).strip()


# Manual overrides for known nickname/legal name mismatches
# Format: {nba_api_normalized_name: bbref_normalized_name}
MANUAL_OVERRIDES = {
    "bones hyland": "vlatko cancar",   # placeholder — update after inspecting unmatched
    "mo bamba": "mohamed bamba",
    "nic claxton": "nicolas claxton",
    "naz reid": "nazreon reid",
    "kenyon martin": "kenyon martin jr",  # note: after suffix removal both become "kenyon martin"
    "tj warren": "t j warren",
    "tj mcconnell": "t j mcconnell",
    "pj tucker": "p j tucker",
    "pj washington": "p j washington",
    "og anunoby": "og anunoby",
    "cam reddish": "cameron reddish",
    "cam thomas": "cameron thomas",
    "cam johnson": "cameron johnson",
    "gary payton": "gary payton ii",  # nba_api drops the II for the son
}

# Apply normalization
game_logs = game_logs.with_columns(
    pl.col("PLAYER_NAME").map_elements(normalize_name, return_dtype=pl.Utf8).alias("PLAYER_NAME_NORM")
)

salaries = salaries.with_columns(
    pl.col("PLAYER_NAME").map_elements(normalize_name, return_dtype=pl.Utf8).alias("PLAYER_NAME_NORM")
)

print(f"Unique normalized names in game logs: {game_logs['PLAYER_NAME_NORM'].n_unique()}")
print(f"Unique normalized names in salaries: {salaries['PLAYER_NAME_NORM'].n_unique()}")

Unique normalized names in game logs: 991
Unique normalized names in salaries: 900


In [45]:
# Apply manual overrides to game_logs names
game_logs = game_logs.with_columns(
    pl.col("PLAYER_NAME_NORM").replace(MANUAL_OVERRIDES).alias("PLAYER_NAME_NORM")
)

# Build lookup: (SEASON, PLAYER_NAME_NORM) → exists in salaries
salary_keys = set(
    zip(salaries["SEASON"].to_list(), salaries["PLAYER_NAME_NORM"].to_list())
)

# Find game log players not found in salaries (for a quick scan)
game_log_keys = (
    game_logs
    .select(["SEASON", "PLAYER_NAME_NORM"])
    .unique()
)

unmatched = game_log_keys.filter(
    pl.struct(["SEASON", "PLAYER_NAME_NORM"])
    .map_elements(
        lambda row: (row["SEASON"], row["PLAYER_NAME_NORM"]) not in salary_keys,
        return_dtype=pl.Boolean,
    )
)

print(f"Unmatched players (game logs not in salaries): {len(unmatched)}")
print("\nSample unmatched (first 30):")
print(unmatched.sort(["SEASON", "PLAYER_NAME_NORM"]).head(30))

Unmatched players (game logs not in salaries): 484

Sample unmatched (first 30):
shape: (30, 2)
┌─────────┬──────────────────┐
│ SEASON  ┆ PLAYER_NAME_NORM │
│ ---     ┆ ---              │
│ str     ┆ str              │
╞═════════╪══════════════════╡
│ 2020-21 ┆ adam mokoka      │
│ 2020-21 ┆ alen smailagic   │
│ 2020-21 ┆ amida brimah     │
│ 2020-21 ┆ amir coffey      │
│ 2020-21 ┆ anderson varejao │
│ …       ┆ …                │
│ 2020-21 ┆ dennis schröder  │
│ 2020-21 ┆ devon dotson     │
│ 2020-21 ┆ devontae cacok   │
│ 2020-21 ┆ didi louzada     │
│ 2020-21 ┆ ersan ilyasova   │
└─────────┴──────────────────┘


In [46]:
# Fuzzy matching fallback for unmatched players
# Build salary lookup per season
salary_by_season = (
    salaries
    .group_by("SEASON")
    .agg(pl.col("PLAYER_NAME_NORM").alias("names"))
)

fuzzy_map = {}  # {(season, unmatched_norm): matched_norm}

for row in salary_by_season.iter_rows(named=True):
    season = row["SEASON"]
    candidates = row["names"]

    unmatched_season = unmatched.filter(pl.col("SEASON") == season)["PLAYER_NAME_NORM"].to_list()

    for name in unmatched_season:
        result = rf_process.extractOne(
            name, candidates,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=88,
        )
        if result:
            fuzzy_map[(season, name)] = result[0]
            print(f"  Fuzzy match [{season}]: '{name}' → '{result[0]}' (score={result[1]})")

print(f"\nFuzzy matches found: {len(fuzzy_map)}")

  Fuzzy match [2020-21]: 'juancho hernangomez' → 'juancho hernangã³mez' (score=92.3076923076923)
  Fuzzy match [2020-21]: 'nicolo melli' → 'nicolã² melli' (score=88.0)
  Fuzzy match [2020-21]: 'p j washington' → 'pj washington' (score=88.88888888888889)
  Fuzzy match [2020-21]: 't j mcconnell' → 'tj mcconnell' (score=88.0)
  Fuzzy match [2020-21]: 'jonas valančiūnas' → 'jonas valanäiånas' (score=88.23529411764706)
  Fuzzy match [2020-21]: 'willy hernangomez' → 'willy hernangã³mez' (score=91.42857142857143)
  Fuzzy match [2020-21]: 'jusuf nurkić' → 'jusuf nurkiä' (score=91.66666666666666)
  Fuzzy match [2020-21]: 'nikola jokić' → 'nikola jokiä' (score=91.66666666666666)
  Fuzzy match [2020-21]: 'anderson varejao' → 'anderson varejão' (score=93.75)
  Fuzzy match [2020-21]: 'alen smailagic' → 'alen smailagiä' (score=92.85714285714286)
  Fuzzy match [2020-21]: 'kristaps porziņģis' → 'kristaps porziåäis' (score=88.88888888888889)
  Fuzzy match [2020-21]: 'dennis schröder' → 'dennis schrãder

In [47]:
# Apply fuzzy map to game_logs
def apply_fuzzy_map(season: str, name: str) -> str:
    return fuzzy_map.get((season, name), name)

if fuzzy_map:
    game_logs = game_logs.with_columns(
        pl.struct(["SEASON", "PLAYER_NAME_NORM"])
        .map_elements(
            lambda row: apply_fuzzy_map(row["SEASON"], row["PLAYER_NAME_NORM"]),
            return_dtype=pl.Utf8,
        )
        .alias("PLAYER_NAME_NORM")
    )
    print("Fuzzy corrections applied.")
else:
    print("No fuzzy corrections needed.")

Fuzzy corrections applied.


In [48]:
# Post-fuzzy check: how many players are still unmatched?
salary_keys_updated = set(
    zip(salaries["SEASON"].to_list(), salaries["PLAYER_NAME_NORM"].to_list())
)

still_unmatched = (
    game_logs
    .select(["SEASON", "PLAYER_NAME_NORM", "PLAYER_NAME"])
    .unique()
    .filter(
        pl.struct(["SEASON", "PLAYER_NAME_NORM"])
        .map_elements(
            lambda row: (row["SEASON"], row["PLAYER_NAME_NORM"]) not in salary_keys_updated,
            return_dtype=pl.Boolean,
        )
    )
    .sort(["SEASON", "PLAYER_NAME_NORM"])
)

print(f"Still unmatched after all corrections: {len(still_unmatched)}")
print("(These players will be dropped by the INNER JOIN — expected: two-way/G-League contracts)")
print()
print(still_unmatched.to_pandas().to_string(index=False))

Still unmatched after all corrections: 398
(These players will be dropped by the INNER JOIN — expected: two-way/G-League contracts)

 SEASON       PLAYER_NAME_NORM            PLAYER_NAME
2020-21            adam mokoka            Adam Mokoka
2020-21           amida brimah           Amida Brimah
2020-21            amir coffey            Amir Coffey
2020-21           anthony lamb           Anthony Lamb
2020-21       anžejs pasečņiks       Anžejs Pasečņiks
2020-21          armoni brooks          Armoni Brooks
2020-21          ashton hagans          Ashton Hagans
2020-21           axel toupane           Axel Toupane
2020-21            brian bowen         Brian Bowen II
2020-21         brodric thomas         Brodric Thomas
2020-21           cam reynolds           Cam Reynolds
2020-21        cameron reddish            Cam Reddish
2020-21        cassius stanley        Cassius Stanley
2020-21        cassius winston        Cassius Winston
2020-21         chasson randle         Chasson Randle
202

## Step 5: SQL Join via DuckDB

We use DuckDB to perform the join using SQL — a course requirement. We join on `(PLAYER_NAME_NORM, SEASON)` using an INNER JOIN, which naturally drops game rows for players without a matched salary (two-way/G-League players).

In [49]:
con = duckdb.connect()
con.register("game_logs_tbl", game_logs)
con.register("salaries_tbl", salaries)

result_df = con.execute("""
    SELECT
        g.*,
        s.SALARY AS SEASON_SALARY,
        s.BBREF_ID
    FROM game_logs_tbl g
    INNER JOIN salaries_tbl s
        ON g.PLAYER_NAME_NORM = s.PLAYER_NAME_NORM
        AND g.SEASON = s.SEASON
""").pl()

print(f"After join: {len(result_df):,} rows × {result_df.width} cols")
print(f"Null SEASON_SALARY count: {result_df['SEASON_SALARY'].null_count()}")
print(f"Unique players after join: {result_df['PLAYER_NAME'].n_unique()}")

After join: 119,227 rows × 34 cols
Null SEASON_SALARY count: 0
Unique players after join: 828


## Step 6: Feature Engineering

Create derived features for modeling:
- **Per-36 stats**: normalize raw stats to 36 minutes of play time
- **USAGE_PROXY**: approximates usage rate without team totals
- **WIN / IS_HOME**: binary game context flags
- **LOG_SALARY**: log-transformed salary for regression (addresses right skew)
- **SALARY_TIER**: NTILE(4) classification target — 'budget', 'mid_low', 'mid_high', 'max'

In [50]:
# Per-36 features
per36_stats = ["PTS", "REB", "AST", "STL", "BLK", "TOV"]
per36_exprs = [
    (pl.col(s).cast(pl.Float32) / pl.col("MIN") * 36).alias(f"{s}_PER36")
    for s in per36_stats
    if s in result_df.columns
]

result_df = result_df.with_columns(
    *per36_exprs,
    # Usage proxy: (FGA + 0.44*FTA + TOV) / MIN
    ((pl.col("FGA") + 0.44 * pl.col("FTA") + pl.col("TOV")) / pl.col("MIN"))
    .alias("USAGE_PROXY"),
    # Win flag
    (pl.col("WL") == "W").cast(pl.Int8).alias("WIN"),
    # Home flag: 'vs.' means home game in nba_api MATCHUP column
    pl.col("MATCHUP").str.contains(r"vs\.").cast(pl.Int8).alias("IS_HOME"),
    # Log salary (regression target)
    pl.col("SEASON_SALARY").log(base=math.e).alias("LOG_SALARY"),
)

print("Per-36, usage, WIN, IS_HOME, LOG_SALARY features added.")
print(f"Shape: {result_df.shape}")

# Join bio data onto result_df via BBREF_ID (from salary CSV) — more reliable than name matching
result_df = result_df.join(
    bios.select(["BBREF_ID", "POSITION_SIMPLE", "BIRTH_DATE_DT", "YEARS_EXP", "HEIGHT", "WEIGHT"]),
    on="BBREF_ID",
    how="left",
)

# Age at game time (years, floating point)
result_df = result_df.with_columns(
    ((pl.col("GAME_DATE") - pl.col("BIRTH_DATE_DT")).dt.total_days() / 365.25)
    .cast(pl.Float32)
    .alias("AGE_AT_GAME"),
)

null_bio = result_df["POSITION_SIMPLE"].null_count()
print(f"Bio features joined. Rows missing position: {null_bio} ({null_bio/len(result_df)*100:.1f}%)")
print(f"Shape: {result_df.shape}")

Per-36, usage, WIN, IS_HOME, LOG_SALARY features added.
Shape: (119227, 44)
Bio features joined. Rows missing position: 21900 (18.4%)
Shape: (119227, 50)


In [52]:
# SALARY_TIER: 4-class classification target via DuckDB NTILE(4) partitioned by season
# Ensures balanced tiers within each season (accounting for salary cap growth)
con.register("featured_tbl", result_df)

result_df = con.execute("""
    SELECT
        *,
        NTILE(4) OVER (PARTITION BY SEASON ORDER BY SEASON_SALARY) AS SALARY_TIER_NUM
    FROM featured_tbl
""").pl()

# Map numeric tier to meaningful labels (replace_strict required for int->str in Polars 1.x)
result_df = result_df.with_columns(
    pl.col("SALARY_TIER_NUM")
    .replace_strict({1: "budget", 2: "mid_low", 3: "mid_high", 4: "max"}, return_dtype=pl.Utf8)
    .alias("SALARY_TIER")
)

print(f"SALARY_TIER value counts:")
print(result_df.group_by("SALARY_TIER").len().sort("SALARY_TIER"))
print(f"Unique tiers: {sorted(result_df['SALARY_TIER'].unique().to_list())}")

SALARY_TIER value counts:
shape: (4, 2)
┌─────────────┬───────┐
│ SALARY_TIER ┆ len   │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ budget      ┆ 29809 │
│ max         ┆ 29804 │
│ mid_high    ┆ 29805 │
│ mid_low     ┆ 29809 │
└─────────────┴───────┘
Unique tiers: ['budget', 'max', 'mid_high', 'mid_low']


## Step 7: Final Validation

In [53]:
print("=== Final Dataset Summary ===")
print(f"Shape: {result_df.shape}")
print(f"Rows: {len(result_df):,} (target: 50,000+)")
print(f"Columns: {result_df.width}")
print(f"\nNull counts (key columns):")
key_cols = ["SEASON_SALARY", "LOG_SALARY", "SALARY_TIER", "MIN", "PTS", "REB", "AST"]
for col in key_cols:
    if col in result_df.columns:
        print(f"  {col}: {result_df[col].null_count()} nulls")

print(f"\nSalary range: ${result_df['SEASON_SALARY'].min():,.0f} — ${result_df['SEASON_SALARY'].max():,.0f}")
print(f"LOG_SALARY range: {result_df['LOG_SALARY'].min():.2f} — {result_df['LOG_SALARY'].max():.2f}")
print(f"\nSALARY_TIER unique: {sorted(result_df['SALARY_TIER'].unique().to_list())}")
assert result_df['SALARY_TIER'].n_unique() == 4, "Expected exactly 4 salary tiers!"
assert result_df['SEASON_SALARY'].null_count() == 0, "SEASON_SALARY has nulls!"
assert len(result_df) >= 50_000, f"Only {len(result_df):,} rows — need 50,000+!"
print("\nAll assertions passed!")

=== Final Dataset Summary ===
Shape: (119227, 52)
Rows: 119,227 (target: 50,000+)
Columns: 52

Null counts (key columns):
  SEASON_SALARY: 0 nulls
  LOG_SALARY: 0 nulls
  SALARY_TIER: 0 nulls
  MIN: 0 nulls
  PTS: 0 nulls
  REB: 0 nulls
  AST: 0 nulls

Salary range: $5,318 — $55,761,216
LOG_SALARY range: 8.58 — 17.84

SALARY_TIER unique: ['budget', 'max', 'mid_high', 'mid_low']

All assertions passed!


## Step 8: Save Processed Data

In [54]:
parquet_path = PROCESSED_DIR / "nba_player_games_clean.parquet"
csv_path = PROCESSED_DIR / "nba_player_games_clean.csv"

result_df.write_parquet(parquet_path)
result_df.write_csv(csv_path)

print(f"Saved Parquet: {parquet_path} ({parquet_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved CSV:     {csv_path} ({csv_path.stat().st_size / 1e6:.1f} MB)")
print(f"\nFinal shape: {result_df.shape}")
print("\nColumn list:")
for i, col in enumerate(result_df.columns):
    print(f"  {i+1:2d}. {col} ({result_df[col].dtype})")

Saved Parquet: ../data/processed/nba_player_games_clean.parquet (5.7 MB)
Saved CSV:     ../data/processed/nba_player_games_clean.csv (38.1 MB)

Final shape: (119227, 52)

Column list:
   1. SEASON (String)
   2. PLAYER_ID (Int64)
   3. PLAYER_NAME (String)
   4. TEAM_ABBREVIATION (String)
   5. GAME_ID (Int64)
   6. GAME_DATE (Date)
   7. MATCHUP (String)
   8. WL (String)
   9. MIN (Float32)
  10. FGM (Float32)
  11. FGA (Float32)
  12. FG_PCT (Float32)
  13. FG3M (Float32)
  14. FG3A (Float32)
  15. FG3_PCT (Float32)
  16. FTM (Float32)
  17. FTA (Float32)
  18. FT_PCT (Float32)
  19. OREB (Float32)
  20. DREB (Float32)
  21. REB (Float32)
  22. AST (Float32)
  23. TOV (Float32)
  24. STL (Float32)
  25. BLK (Float32)
  26. BLKA (Float32)
  27. PF (Float32)
  28. PFD (Float32)
  29. PTS (Float32)
  30. PLUS_MINUS (Float32)
  31. NBA_FANTASY_PTS (Float32)
  32. PLAYER_NAME_NORM (String)
  33. SEASON_SALARY (Float64)
  34. BBREF_ID (String)
  35. PTS_PER36 (Float32)
  36. REB_PER36 (Fl

## Summary

The cleaned dataset has been saved to `../data/processed/nba_player_games_clean.parquet`.

**Key decisions made in this notebook:**
- Filtered DNP rows (`MIN > 0`) to ensure all rows represent actual game appearances
- Used INNER JOIN to naturally exclude two-way/G-League players with no BBRef salary record
- SALARY_TIER is partitioned **by season** so that each season independently has 4 equal-sized tiers, accounting for year-over-year salary cap growth
- LOG_SALARY is the regression target (raw salary is highly right-skewed)

Proceed to `03_eda.ipynb` for exploratory analysis.